In [ ]:
# Building a CNN model to categorize a persons head shape into four different categories: oval, round, heart, square 
# using python version: 3.10.13 to ensure compatibility with mediapipe
import os
import mediapipe as mp
import cv2
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
image_dir = 'src/data/raw_images' 

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5)

# Function to extract face mesh landmarks
def extract_face_landmarks(image_path):
    try:
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not read image: {image_path}")
            return None
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)
        if results.multi_face_landmarks:
            landmarks = []
            for landmark in results.multi_face_landmarks[0].landmark:
                # Extract x, y, z coordinates (normalized to image dimensions)
                # We'll primarily use x and y for clustering shape
                landmarks.extend([landmark.x, landmark.y]) # , landmark.z]) # Including z might add complexity, start with x, y
            return np.array(landmarks)
        else:
            print(f"No face detected in: {image_path}")
            return None
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None
    
# Load and process images
image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

all_landmarks = []
valid_image_files = []
print(f"Found {len(image_files)} image files.")

for i, img_file in enumerate(image_files):
    print(f"Processing image {i+1}/{len(image_files)}: {img_file}")
    landmarks = extract_face_landmarks(img_file)
    if landmarks is not None:
        all_landmarks.append(landmarks)
        valid_image_files.append(img_file)

# Ensure we have data to cluster
if not all_landmarks:
    print("No valid face landmarks extracted from any image. Exiting.")
else:
    print(f"Successfully extracted landmarks from {len(all_landmarks)} images.")
    # Convert list of landmark arrays to a single numpy array
    X = np.array(all_landmarks)

    # Apply K-Means clustering
    # You can experiment with the number of clusters (n_clusters) if 4 is not ideal based on results
    n_clusters = 4
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) # Added n_init
    kmeans.fit(X)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_

    # Analyze and interpret the clusters
    # The clusters represent different groupings of face landmark features.
    # You can examine the cluster centers or look at example images from each cluster
    # to try and correlate them with the desired head shapes.

    print("\nCluster assignments for each image:")
    for i, img_file in enumerate(valid_image_files):
        print(f"{img_file}: Cluster {labels[i]}")

    # Optional: Visualize the cluster centers (though interpreting them as shapes is complex)
    # This requires dimensionality reduction if your landmark vector is high dimensional.
    # For simplicity, let's just show how many images are in each cluster
    print("\nNumber of images per cluster:")
    for i in range(n_clusters):
        print(f"Cluster {i}: {np.sum(labels == i)} images")

    # To further understand the clusters, you could:
    # 1. Pick representative images from each cluster and visually inspect their head shapes.
    # 2. (More advanced) Perform dimensionality reduction (e.g., PCA, t-SNE) on the landmark data
    #    to visualize the clusters in 2D or 3D space.

    # Example of getting images for a specific cluster (e.g., Cluster 0)
    print("\nExample images from Cluster 0:")
    cluster_0_indices = np.where(labels == 0)[0]
    for i in cluster_0_indices[:5]: # Print up to 5 example images from Cluster 0
        print(valid_image_files[i])

# Release resources
face_mesh.close()


In [ ]:
# Visual Inspection: Display Clustered Images
for cluster_id in range(n_clusters):
    print(f"\nShowing sample images from Cluster {cluster_id}")
    idxs = np.where(labels == cluster_id)[0][:10]  # Show 5 examples
    plt.figure(figsize=(20, 2))
    for i, idx in enumerate(idxs):
        img = cv2.imread(valid_image_files[idx])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, 10, i + 1)
        plt.imshow(img_rgb)
        plt.title(f"Img {i+1}")
        plt.axis('off')
    plt.suptitle(f"Cluster {cluster_id} - Sample Images")
    plt.tight_layout()
    plt.show()

In [ ]:
# Export Cluster Results as JSON
import re
import json

def extract_number(filename):
    """Extract the number from filenames like face-3747.jpg"""
    match = re.search(r"(\d+)", filename)
    return int(match.group(1)) if match else float('inf')

cluster_results = []

for i, img_path in enumerate(valid_image_files):
    cluster_results.append({
        "image": os.path.basename(img_path),
        "cluster": int(labels[i])
    })

# Sort results by image number
cluster_results_sorted = sorted(cluster_results, key=lambda x: extract_number(x["image"]))

# Save to JSON file in current directory
json_path = "src/data/cluster_results.json"
with open(json_path, 'w') as f:
    json.dump(cluster_results_sorted, f, indent=4)

print(f"\nSorted cluster assignments saved to: {json_path}")
